# Battery dispatch optimization
The goal of this exercise is to optimize the charge/discharge behavior of a battery system performing energy arbitrage in the NYISO market. The model objective is to maximize revenue over a year and the project objective is to gain insights into market dynamics and expected system behavior. We have hourly LBMP data for the entire year (taken from 2017), which we assume is accurate with perfect foresight.

The battery system has maximum storage capacity of 200 kWh and a power rating of 100 kW (charge and discharge). Round-trip AC-AC efficiency is 85%. The maximum daily discharge throughput is constrained to 200 kWh within a 24-hour period.

Because we are using this model to understand the market dynamics in the NYISO NYC hub, it's fine to run the entire year optimzation at once rather than running in discrete periods. When actually operating the battery we would need to use price forecasts that would become less accurate over time. In that situation a series of multi-day optimizations would be used. The first window would assume a starting charge state (I'm using half of the manimum charge in this exercise), and might predict 3-4 days. The first 24-hours would be kept and used as a constraint in the next period. For the second period, a window of 4-5 days would be used -- values from the first day would be used to constrain hours 0-23, and the optimization would again use a look-ahead of 2-3 days to ensure accurate model behavior. This cycle (keep 24 hours and use then as a constrain in the next iteration) would then continue through the end of the year.

I've elected to stick with the simplier approach here because it is easier/faster to code and should provide the same optimized dispatch results. Actually optimizing dispatch for a battery system would require forecasting prices. Because this is a much simpler exercise a simpler model that takes less time to build is fine.

In [ ]:
from IPython.display import clear_output, display, HTML


# Clear all outputs
clear_output(wait=True)

# Clear all variables
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('reset -f')
except:
    pass

# Clear console (optional)
import os
os.system('clear')  # for Mac/Linux

import pandas as pd
from pathlib import Path
import altair as alt

# Select your appropriate notebook type for rendering Altair figures
alt.renderers.enable('jupyterlab')
# alt.renderers.enable('notebook')
alt.data_transformers.enable('default', max_rows=None)

from src.read_data import read_all_nyc
from src.read_all_NL import read_all_NL
from src.battery_model import optimize_year, model_to_df

## Read data
Functions to read data and run the optimization model are provided in scripts in the `src` folder. The `read_all_nyc` function combines data from daily .csv files, filters out all non-NYC node prices, and renames the columns to snake case.

In [ ]:
# data_path = Path.cwd() / '2017_NYISO_LBMPs'
# df = read_all_nyc(data_path=data_path)

#data_path = Path.cwd() / 'Entsoe_prices_2023'
#df = read_entsoe_2023(data_path=data_path)

df = read_all_NL()

# copy dataframe originally imported from .csv  
df_import = df.copy()

df_import.tail()    

Selecting a portion of the dataframe - to reduce test run time


In [ ]:
# start point:
n = 0 
first_model_hour = n
df = df[n:]
print("n = ", n)

# end point:
portion = 1.00
m = int(portion * len(df_import))
last_model_hour = m-1
df = df[:m]
print("last_model_hour = ", last_model_hour)

In [ ]:
df.head()

In [ ]:
df.tail()

## Model parameters and constraints

**Parameters**
- $t$: timestep or hour
- $R_{max}$ (100 kW): maximum power than can be delivered to or from the battery (charge or discharge rate)
- $S_{max}$ (200 kWh): maximum battery capacity
- $S_t$: storage at time $t$
- Eff ($\eta$) (85%): efficiency 
- $D_{max}$ (200 kWh): max discharge within a 24 hour period
- $P_t$: LBMP at time $t$

**Decision variables**
- $E^{in}_t$: energy delivered to the battery at time $t$
- $E^{out}_t$: energy discharged from the battery at time $t$

**Constraints**
- $S_1$ = $\frac{S_{max}}{2}$ (Assume storage begins at half of capacity)
- $S_t$ = $S_{t-1} + \sqrt{\eta} \times E^{in}_{t-1} - \frac{E^{out}_{t-1}}{\sqrt{\eta}}$
- $\forall t, S_t \geq 0$
- $\forall t, S_t \leq S_{max}$
- $\forall t, E^{in}_t \leq R_{max}$
- $\forall t, E^{out}_t \leq R_{max}$
- $\forall t, E^{out}_t \leq S_t$
- $\sum_{t'=t-23}^t E^{out}_{t'} \leq D_{max} \forall t \subset (T, t \geq 24)$

## Run the optimization model
The `optimize_year` function takes in the LBMP data from our new dataframe and returns the optimization results in a dataframe.

In [ ]:
print(df.iloc[n,:])

print("\n length of remaining dataframe is: ", m)
print(df.iloc[m-1,:])

# Adjust the indices to be within the valid range
results_df = optimize_year(df, first_model_hour, last_model_hour)

In [ ]:
import pytz

# Ensure 'time_stamp' is in datetime format, and if it's timezone-aware, convert it to UTC first
results_df['time_stamp'] = pd.to_datetime(results_df['time_stamp'], utc=True)

# Convert to Amsterdam timezone
amsterdam_tz = pytz.timezone('Europe/Amsterdam')
results_df['time_stamp'] = results_df['time_stamp'].dt.tz_convert(amsterdam_tz)

results_df.head()

In [ ]:
results_df.tail()

## Analysis of results
In this exercise I've been asked to present the following:
- Summary values
    - Annual revenue
    - Annual charging costs
    - Annual discharged throughput
- Plots
    - Hourly dispatch and LBMP for the most profitable week (assuming calendar week)
    - Total profit for each month

### Summary values
Revenue, costs, and profit still need to be calculated using energy in/out and the hourly price

In [9]:
results_df['revenue'] = results_df.Eout * results_df.lbmp / 1000
results_df['charge_cost'] = results_df.Ein * results_df.lbmp  / 1000
results_df['profit'] = results_df.revenue - results_df.charge_cost

In [ ]:
total_revenue       = results_df.revenue.sum()
total_charge_cost   = results_df.charge_cost.sum()
total_profit        = results_df.profit.sum()
total_charge        = results_df.Ein.sum()
total_discharge     = results_df.Eout.sum()

total_charge_losses = results_df.Ein.sum() - results_df.Eout.sum()

Rmax = (max(results_df.Ein) * 4)        # Battery power (max charged)
Smax = max(results_df.charge_state)     # Battery capacity (max stored)
print('Battery Capacity 100% nett SOC: {:,.0f} kWh'.format(Smax))

total_profit_per_1MW = total_profit / Rmax * 1000
total_profit_per_1MWh = total_profit / Smax * 1000

print('Annual profit was €{:,.0f} per MWh'.format(total_profit_per_1MWh))
print('Annual profit was €{:,.0f} per MW'.format(total_profit_per_1MW))


BESS_CAPEX = 300 * 1000 # €/MWh
BESS_cycle_life = 6000

Batt_investment = BESS_CAPEX * Smax / 1000
payback_time = round(total_profit / Batt_investment, 1)

cycle_depreciation_cost = BESS_CAPEX / BESS_cycle_life 
print('cycle_depreciation_cost: {:,.0f} €/MWh'.format(cycle_depreciation_cost))

cycles_made = total_charge/Smax
print('Cycles per year: {:,.0f}'.format(cycles_made))

cycle_depreciation_cost_for_1MWh = cycle_depreciation_cost * cycles_made
print('cycle_depreciation_cost for cycling 1 MWh per year: €{:,.0f}'.format(cycle_depreciation_cost_for_1MWh))

depreciation_cost = cycle_depreciation_cost * total_charge / 1000


print("\n")
print('Annual profit was €{:,.0f}'.format(total_profit), '(battery size being {:,.0f}kWh'.format(Smax), 'with {:,.0f}kW max charge)'.format(Rmax))
print('Annual discharge income was €{:,.0f}'.format(total_revenue))
print('Annual charging cost was €{:,.0f}'.format(total_charge_cost))
print('depreciation_cost: €{:,.0f}'.format(depreciation_cost))
print('Annual discharged throughput was {:,.0f} kWh'.format(total_discharge))

avg_charge_price = total_charge_cost / total_charge *1000
avg_discharge_price = total_revenue / total_discharge *1000
avg_charge_losses = total_charge_losses / total_charge

avg_spread_price = avg_discharge_price - avg_charge_price
avg_profit_price = total_profit / total_charge *1000

print("\n")
print('Annual average charged price was {:,.0f} €/MWh'.format(avg_charge_price))
print('Annual average discharged price was {:,.0f} €/MWh'.format(avg_discharge_price))
print('Annual average charge losses was {:,.0f}%'.format(avg_charge_losses*100))
print('Annual average spread price was {:,.0f} €/MWh'.format(avg_spread_price))
print('Annual average profit price was {:,.0f} €/MWh'.format(avg_profit_price))

year_used = int(df['time_stamp'].dt.year.iloc[0])

# Save parameters to CSV
parameters = {
    'Year_price_data': [year_used],
    'Elec_price': ['NL'],
    'Dispatch_strategy': ['max_profit GPLK optimization, 98 percent perfect solution, real world can be 10-20% lower profit'],
    'Battery_capacity_kWh': [int(Smax)],
    'Battery_power_kW': [int(Rmax)],
    'Battery_CAPEX_investment': [int(Batt_investment)],
    'annual_revenue': [int(total_revenue)],
    'annual_charge_cost': [int(total_charge_cost)],
    'annual_profit': [int(total_profit)],
    'payback_time': [(payback_time)],
    'annual_charge': [int(total_charge)],
    'annual_discharge': [int(total_discharge)],
    'annual_charge_losses': [int(total_charge_losses)],
    'cycle_depreciation_cost_Euro_per_MWh': [int(cycle_depreciation_cost)],
    'cycles_made_per_year': [int(cycles_made)],
    'annual_depreciation_cost': [int(depreciation_cost)],
    'avg_charge_price_Euro_per_MWh': [(avg_charge_price)],
    'avg_discharge_price_Euro_per_MWh': [(avg_discharge_price)],
    'avg_charge_losses_percent': [(avg_charge_losses)],
    'avg_spread_price_Euro_per_MWh': [(avg_spread_price)],
    'avg_profit_price_Euro_per_MWh': [(avg_profit_price)],
    'cycle_depreciation_cost_for_1MWh': [int(cycle_depreciation_cost_for_1MWh)],
    'annual_profit_per_1MWh_battery_capacity': [int(total_profit_per_1MWh)],
    'total_profit_per_1MW_battery_power': [int(total_profit_per_1MW)]
}

parameters_df = pd.DataFrame(parameters)
filename = f'parameters_summary_{year_used}_{int(cycles_made)}_cycles'
parameters_df.T.to_csv(f'{filename}.csv', header=False)
parameters_df.T.to_excel(f'{filename}.xlsx', header=False)

#TODO: delete 'regeltoestand-2' uren 
#DONE by setting to €50/MWh hard coded, quick-fix

### Output results

In [11]:
results_df.to_csv(f'full_optimization_results_{year_used}_{int(cycles_made)}_cycles.csv')

### Figures

In [12]:
results_df['day'] = results_df.time_stamp.dt.isocalendar().day
results_df['week'] = results_df.time_stamp.dt.isocalendar().week
results_df['month'] = results_df.time_stamp.dt.month
results_df['hour_of_day'] = results_df.time_stamp.dt.hour


Including both the dispatch and hourly Day-Ahead EPEX prices (in NY:LBMP) in a single plot is difficult because their values have different scales (dispatch is capped at 100 kWh while LBMP in week 52 goes over \\$200/kWh). A dual y-axis plot can be difficult to read, so I've decided to show one plot on top of the other. The plot of LBMP uses color to encode dispatch, which helps to make the whole thing easier to interprete.

In [ ]:
data = results_df.loc[(results_df.time_stamp >= '2022-03-01') &
                      (results_df.time_stamp <= '2024-01-10'), :].copy()

data.loc[:, 'dispatch'] = data.Ein - data.Eout

dispatch_data = pd.melt(data, id_vars='time_stamp', value_vars=['Eout', 'Ein'], var_name='Dispatch')

color_scale = alt.Scale(
            domain=['Ein', 'Eout'],
            range=['#f99820', '#2081f9']
        )

dispatch = alt.Chart(dispatch_data).mark_line().encode(
    x='time_stamp:T',
    y=alt.Y('value:Q', axis=alt.Axis(title='Electricity in/out (kWh)')),
    color=alt.Color('Dispatch:N', scale=color_scale)
).properties(
    width=1000,
    height=300
)

lbmp = alt.Chart(data).mark_circle().encode(
    x='time_stamp:T',
    y=alt.Y('lbmp:Q', axis=alt.Axis(title='NL Day-Ahead €/MWh')),
    color=alt.Color('dispatch:Q', scale=alt.Scale(scheme='blueorange')),
    tooltip='dispatch:Q'
).properties(
    width=1000,
    height=300
)

charge_state = alt.Chart(data).mark_line().encode(
    x='time_stamp:T',
    y=alt.Y('charge_state:Q', axis=alt.Axis(title='Charge State (kWh)')),
    color=alt.value('green')
).properties(
    width=1000,
    height=300
)

alt.vconcat(
    dispatch,
    lbmp,
    charge_state
)

In [ ]:
import plotly.express as px

# Create a custom color scale
color_scale = [
    [0, 'red'],  # Grey for dispatch = 0
    [0.5, 'grey'],  # Blue for negative dispatch
    [1, 'blue']  # Red for positive dispatch
]

# Assuming 'data' is your pandas DataFrame
fig = px.scatter(
    data, 
    x='time_stamp', 
    y='lbmp', 
    color='dispatch',
    color_continuous_scale=color_scale,  # Use the custom color scale
    labels={'lbmp': 'NL Day-Ahead €/MWh'},
    title='Scatter Plot of LBMP vs Time',
)

# Adjust layout for better readability
fig.update_layout(
    height=600,  # Adjust the height of the plot
)

# Show the plot
fig.show()

In [ ]:
# Assuming 'results_df' is your DataFrame
alt.Chart(results_df).mark_bar().encode(
    x=alt.X('month:O', axis=alt.Axis(title='Month')),  # x-axis label for month
    y=alt.Y('sum(profit):Q', axis=alt.Axis(title='Total Profit (€)'))  # y-axis label for profit
).properties(
    height=300,
    width=500,
)


In [ ]:
# Assuming 'results_df' is your DataFrame
#results_df['day'] = results_df['time_stamp'].dt.date
##
alt.Chart(results_df).mark_bar().encode(
    x=alt.X('day:O', axis=alt.Axis(title='Day')),  # x-axis label for week
    y=alt.Y('sum(profit):Q', axis=alt.Axis(title='Total Profit (€)'))  # y-axis label for profit
).properties(
    height=300,
    width=500,
)
